# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Mohil Ahuja  
**`Roll Number`:** U20230121 
**`GitHub Branch`:** Mohil_U20230121  

# Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from rlcmab_sampler import sampler


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\mohil\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



# Load Datasets

In [2]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print("=== News Articles ===")
print(f"Shape: {news_df.shape}")
print(f"Columns: {news_df.columns.tolist()}")
print(news_df.head())

print("\n=== Train Users ===")
print(f"Shape: {train_users.shape}")
print(f"Columns: {train_users.columns.tolist()}")
print(train_users.head())

print("\n=== Test Users ===")
print(f"Shape: {test_users.shape}")
print(test_users.head())

=== News Articles ===
Shape: (209527, 6)
Columns: ['link', 'headline', 'category', 'short_description', 'authors', 'date']
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [3]:
# --- Data Preprocessing ---

# 1. Check for missing values
print("Missing values in train_users:")
print(train_users.isnull().sum()[train_users.isnull().sum() > 0])
print(f"\nMissing values in test_users:")
print(test_users.isnull().sum()[test_users.isnull().sum() > 0])

# 2. Columns to drop (non-predictive or high-cardinality)
drop_cols = ['user_id', 'label', 'browser_version']

# 3. Build feature matrix
def preprocess(df, fit=False):
    X = df.copy()
    
    # Fill missing age with median
    if fit:
        preprocess.age_median = X['age'].median()
    X['age'] = X['age'].fillna(preprocess.age_median)
    
    # Encode region_code via frequency encoding (handles unseen values)
    if fit:
        preprocess.region_freq = X['region_code'].value_counts(normalize=True).to_dict()
    X['region_code'] = X['region_code'].map(preprocess.region_freq).fillna(0.0)
    
    # Encode subscriber as int
    X['subscriber'] = X['subscriber'].astype(int)
    
    # Drop non-feature columns
    cols_to_drop = [c for c in drop_cols if c in X.columns]
    X = X.drop(columns=cols_to_drop)
    
    feature_names = X.columns.tolist()
    
    # Scale features
    if fit:
        preprocess.scaler = StandardScaler()
        X_scaled = preprocess.scaler.fit_transform(X)
    else:
        X_scaled = preprocess.scaler.transform(X)
    
    return X_scaled, feature_names

# 4. Apply preprocessing on full train set (fit scaler/encoders here)
X_all, feature_names = preprocess(train_users, fit=True)
y_all = LabelEncoder().fit_transform(train_users['label'])

# 5. 80/20 train-validation split (as per assignment spec)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# 6. Preprocess test set (no labels available)
X_test, _ = preprocess(test_users, fit=False)

# 7. Encode target labels
le = LabelEncoder()
le.fit(train_users['label'])

print(f"\nLabel mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Full training set:  {X_all.shape[0]} samples")
print(f"  Train split (80%): {X_train.shape[0]} samples")
print(f"  Val split (20%):   {X_val.shape[0]} samples")
print(f"Test set:            {X_test.shape[0]} samples")

Missing values in train_users:
age    698
dtype: int64

Missing values in test_users:
age    679
dtype: int64

Label mapping: {'user_1': np.int64(0), 'user_2': np.int64(1), 'user_3': np.int64(2)}
Features (30): ['age', 'income', 'clicks', 'purchase_amount', 'session_duration', 'content_variety', 'engagement_score', 'num_transactions', 'avg_monthly_spend', 'avg_cart_value', 'browsing_depth', 'revisit_rate', 'scroll_activity', 'time_on_site', 'interaction_count', 'preferred_price_range', 'discount_usage_rate', 'wishlist_size', 'product_views', 'repeat_purchase_gap (days)', 'churn_risk_score', 'loyalty_index', 'screen_brightness', 'battery_percentage', 'cart_abandonment_count', 'background_app_count', 'session_inactivity_duration', 'network_jitter', 'region_code', 'subscriber']
Full training set:  2000 samples
  Train split (80%): 1600 samples
  Val split (20%):   400 samples
Test set:            2000 samples


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [4]:
# --- User Classification ---
# Train a Gradient Boosting classifier to predict user category (context)

clf = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

# Train on 80% training split
clf.fit(X_train, y_train)

# --- Evaluate on 20% validation split ---
y_val_pred = clf.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy (20% split): {val_acc:.4f}")
print("\n--- Classification Report (Validation Set) ---")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

# Training accuracy
y_train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy (80% split): {train_acc:.4f}")

# --- Retrain on full training set for best context detection on test_users ---
clf.fit(X_all, le.transform(train_users['label']))

# Predict on test_users (context detection for bandit)
y_test_pred = clf.predict(X_test)
test_users['predicted_label'] = le.inverse_transform(y_test_pred)

print(f"\nTest set predictions distribution:")
print(test_users['predicted_label'].value_counts())

# Feature importance (top 10)
importances = clf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("\nTop 10 Feature Importances:")
for i in sorted_idx[:10]:
    print(f"  {feature_names[i]:30s}: {importances[i]:.4f}")

Validation Accuracy (20% split): 0.9150

--- Classification Report (Validation Set) ---
              precision    recall  f1-score   support

      user_1       0.89      0.87      0.88       142
      user_2       0.98      0.89      0.94       142
      user_3       0.88      0.99      0.93       116

    accuracy                           0.92       400
   macro avg       0.92      0.92      0.92       400
weighted avg       0.92      0.92      0.91       400

Training Accuracy (80% split): 1.0000

Test set predictions distribution:
predicted_label
user_2    696
user_1    663
user_3    641
Name: count, dtype: int64

Top 10 Feature Importances:
  region_code                   : 0.4373
  session_duration              : 0.3441
  scroll_activity               : 0.0171
  preferred_price_range         : 0.0158
  churn_risk_score              : 0.0123
  content_variety               : 0.0122
  income                        : 0.0113
  session_inactivity_duration   : 0.0106
  battery_percen

# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
